In [ ]:
import concurrent.futures
import json
import logging
import os
import sys
import threading

from pathlib import Path

basepath = Path(
    r"C:\Users\s158699\Documents\PhD\Experiments\Simulation\aggregated_event_data"
)

sys.path.append(os.path.join(*basepath.parts))

from aggregated_event_data import simulate
from numpy import isclose
from pathlib import Path

logging.basicConfig(level=logging.WARNING, force=True)
logger = logging.getLogger()

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
    load_rdf_graph,
)
from aggregated_traces.utils.ekg_analysis import (
    compute_aggregation_uniformity,
    compute_aggregation_average_kl_trace_graph,
    compute_trace_probabilities,
    compute_number_of_merges_in_trace_graph,
    compute_number_of_steps_at_aggregations,
)

In [ ]:
from rdflib import Graph, Literal, URIRef, Variable
from typing import List


def compute_trace_results(
    event_data_file: str,
    location: URIRef,
    window_start: Literal,
    window_end: Literal,
    result_files: Path,
    run_file_name: str,
) -> str | None:
    # Parse data to RDF graph
    ekg_graph_rdf = load_rdf_graph(event_data_file, store="Oxigraph")

    # Insert DF/DP relations
    ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

    # Check incoming vs outgoing amount
    # query_result_incoming_outgoing = check_quantities(ekg_graph_rdf)

    # Insert fractions on the relations
    ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

    # Create NetworkX graph
    ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)
    print("ekg_graph_nx")

    # Compute forward trace probabilities
    try:
        df_forward, edges_paths_forward = compute_trace_probabilities(
            rdf_trace_graph=ekg_graph_rdf,
            nx_trace_graph=ekg_graph_nx,
            source_entities_time=[
                (
                    location,
                    (window_start, window_end),
                )
            ],
            trace_backward=False,
        )
    except RuntimeError as e:
        logger.warning(e)
        return None
    print("df_forward", len(df_forward))

    # Compute statistics on trace graphs for source-target node pairs
    aggregation_kl = compute_aggregation_uniformity(graph=ekg_graph_rdf)
    for i in df_forward.index:
        # Compute number of merges in the trace graph
        df_forward.loc[i, "n_merges"] = compute_number_of_merges_in_trace_graph(
            trace_graph=ekg_graph_nx,
            source=df_forward.loc[i, "node_source"],
            target=df_forward.loc[i, "node_target"],
            backward=False,
        )

        # Compute uniformity of splits and merges
        df_forward.loc[i, "split_merge_kl"] = (
            compute_aggregation_average_kl_trace_graph(
                trace_graph=ekg_graph_nx,
                aggregation_kl=aggregation_kl,
                source=df_forward.loc[i, "node_source"],
                target=df_forward.loc[i, "node_target"],
                backward=False,
            )
        )
        print("df_forward", i)

    # Get the number of steps executed at the aggregation events
    steps_aggregations = compute_number_of_steps_at_aggregations(
        graph=ekg_graph_rdf, events=df_forward["node_source"].unique()
    )
    df_forward["n_production_steps_aggregations"] = df_forward["node_source"].map(
        steps_aggregations
    )
    print("n_production_steps_aggregations", i)

    # Validate if probabilities add up to one
    if not all(
        isclose(
            df_forward.groupby(["entity_source", "node_source"])["probability"]
            .sum()
            .astype(float),
            1,
        )
    ):
        raise Warning("Sum of probabilities by production step do not add up to one!")

    df_forward["probability"] = df_forward["probability"].astype(float)
    with lock:
        with open(result_files.joinpath(run_file_name), "w") as f:
            json.dump(
                df_forward.to_dict(orient="records"),
                f,
            )

    ekg_graph_rdf.close()

    return result_files.joinpath(run_file_name)

In [ ]:
from collections import defaultdict
from typing import Dict, Tuple


def generate_scenario(
    output_file: str,
    n_lots: int,
    n_devices: List[int],
    steps_resources: Dict[str, Tuple[int]],
    root_cause_resource: str,
    required_material: Dict[str, str] = None,
    merge_after_steps: List[str] = [],
    # n_lots_merge: int = 2,
    split_configs: List[dict] = [],
) -> str:
    # Select lots for merge after a certain step
    merge_configs = [{"after_step": step} for step in merge_after_steps]
    merge_configs_lots = defaultdict(list)
    for config in merge_configs:
        # for i in [i for i in range(0, n_lots, n_lots_merge)]:
        for i in range(n_lots):
            merge_configs_lots[f"Lot{i}"].append(config)

    production_lots = []
    for i in range(n_lots):
        lot_id = f"Lot{i}"
        production_lot = {
            "id": lot_id,
            "steps": list(steps_resources.keys()),
            "merge": merge_configs_lots.get(lot_id, []),
            "split": split_configs,
            "n_devices": n_devices[i],
        }

        if required_material:
            production_lot["required_material"] = required_material

        production_lots.append(production_lot)

    production_resources = []
    for step, (n, mean_duration) in steps_resources.items():
        production_resources.extend(
            [
                {
                    "id": f"{step}{i}",
                    "step": step,
                    "mean_move": 0.5,
                    "mean_duration": mean_duration,
                    "mean_breakdown": 5,
                    "mean_repair": 10,
                    "process_yield": 0.5 if root_cause_resource == f"{step}{i}" else 1,
                }
                for i in range(n)
            ]
        )

    simulation_config = {
        "production_lots": production_lots,
        "production_resources": production_resources,
        "material_lot_size": 100,
        "packing_unit_size": 50,
    }

    # with lock:
    with open(output_file, "w") as f:
        json.dump(simulation_config, f, indent=2)

    return json.dumps(simulation_config)

# Generate (simulate) and process event logs

In [ ]:
import pyDOE2

from itertools import chain, combinations, product


def all_subsets(iterable):
    "Returns all possible subsets of a given iterable"
    s = list(iterable)
    return list(chain.from_iterable(combinations(s, r) for r in range(len(s) + 1)))


def generate_non_overlapping_pairs(subsets):
    pairs = []
    for subset1, subset2 in product(subsets, subsets):
        if not set(subset1) & set(subset2):  # Ensure no overlapping values
            pairs.append((subset1, subset2))
    return pairs


steps = ["WT", "DB", "WB", "Marking", "FT"]
subsets_of_steps = all_subsets(steps)
split_merge_after_steps_values = generate_non_overlapping_pairs(subsets_of_steps)

number_of_devices_values = [
    [50] * 32,
    [25, 75] * 16,
    [10, 90] * 16,
    [25, 25, 25, 125] * 8,
    [10, 10, 10, 170] * 8,
]

n_resources_factor_values = [
    1,
    2,
    4,
    8,
    16,
]

# Generate Design of Experiments (DoE) with reduction
levels = [
    len(split_merge_after_steps_values),
    len(number_of_devices_values),
    len(n_resources_factor_values),
]
reduction = 5

doe = pyDOE2.gsd(levels, reduction)

print("Number of experiments: ", len(doe))

DataFrame(
    doe, columns=["split+merge_after_steps", "number_of_devices", "n_resources_factor"]
).to_json(f"forward/doe_{len(doe)}.json")

### Local (multiple threads)

In [ ]:
def worker(
    scenario_name, event_log_files, i, location, window_start, window_end, result_files
):
    run_file_name = f"run_{i}.json"
    event_data_file = event_log_files.joinpath(run_file_name)
    simulate.main(
        config_file=scenario_name + ".json",
        runtime=10000,
        output_event_log_file=event_data_file,
    )

    print(event_data_file)

    compute_trace_results(
        event_data_file, location, window_start, window_end, result_files, run_file_name
    )

    return scenario_name


n_runs = 10

# Define the problematic machine
location = URIRef("http://example.org/id/ekg/aggregated_traces/WB1")
window_start = Literal(100)
window_end = Literal(500)

lock = threading.Lock()

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    for i in range(len(doe)):
        merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
        number_of_devices = number_of_devices_values[doe[i][1]]
        n_resources_factor = n_resources_factor_values[doe[i][2]]

        number_of_devices_label = set(str(n) for n in number_of_devices)
        scenario_name = (
            "forward/scenarios/doe"
            + f"merge_after_steps={'-'.join(merge_after_steps)}"
            + f"+split_after_steps={'-'.join(split_after_steps)}"
            + f"+number_of_devices={'-'.join(sorted(number_of_devices_label))}"
            + f"+n_resources_factor={n_resources_factor}"
        )

        split_configs = [
            {"after_step": step, "number_of_split_lots": 2}
            for step in split_after_steps
        ]

        generate_scenario(
            output_file=scenario_name + ".json",
            n_lots=32,
            n_devices=number_of_devices,
            steps_resources={
                "WT": (n_resources_factor * 2, 1),
                "DB": (n_resources_factor * 2, 2),
                "WB": (n_resources_factor * 4, 20),
                "Marking": (n_resources_factor * 4, 1),
                "FT": (n_resources_factor * 2, 1),
            },
            root_cause_resource="WB1",
            # required_material: Dict[str, str] = None,
            merge_after_steps=list(merge_after_steps),
            # n_lots_merge: int = 2,
            split_configs=split_configs,
        )

        event_log_files = basepath.joinpath(f"logs/{scenario_name}/")
        if not event_log_files.exists():
            os.mkdir(event_log_files)

        result_files = Path(f"output/{scenario_name}/")
        if not result_files.exists():
            os.mkdir(result_files)

        future_to_url = {
            executor.submit(
                worker,
                scenario_name,
                event_log_files,
                i,
                location,
                window_start,
                window_end,
                result_files,
            ): i
            for i in range(n_runs)
        }
        for future in concurrent.futures.as_completed(future_to_url):
            i = future_to_url[future]
            try:
                name = future.result()
            except Exception as exception:
                print(f"{event_log_files} run_{i} - {exception}")
            else:
                print(f"{event_log_files} run_{i} - {name}")

In [ ]:
for file in Path(f"output/forward").glob("**/*.json"):
    with open(file, "rb") as f:
        file_gz = str(file).replace(".json", ".json.gz")
        with gzip.open(file_gz, "w") as f_gz:
            f_gz.write(f.read())

### Invoke Lambda function

In [ ]:
!aws sso login --profile=dev-poweruser

In [ ]:
import boto3

boto3.setup_default_session(profile_name="dev-poweruser")
lambda_client = boto3.client("lambda")

n_runs = 10

# Define the fixed parameters
location = "http://example.org/id/ekg/aggregated_traces/WB1"
window_start = 100
window_end = 500


for i in range(len(doe)):
    merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
    number_of_devices = number_of_devices_values[doe[i][1]]
    n_resources_factor = n_resources_factor_values[doe[i][2]]

    number_of_devices_label = set(str(n) for n in number_of_devices)
    scenario_name = (
        "forward/scenarios/doe/"
        + f"merge_after_steps={'-'.join(merge_after_steps)}"
        + f"+split_after_steps={'-'.join(split_after_steps)}"
        + f"+number_of_devices={'-'.join(sorted(number_of_devices_label))}"
        + f"+n_resources_factor={n_resources_factor}"
    )

    split_configs = [
        {"after_step": step, "number_of_split_lots": 2} for step in split_after_steps
    ]

    event = {
        "quality_threshold": 0.9,
        "scenario_name": scenario_name,
        "scenario": generate_scenario(
            output_file=scenario_name + ".json",
            n_lots=32,
            n_devices=number_of_devices,
            steps_resources={
                "WT": (n_resources_factor * 2, 1),
                "DB": (n_resources_factor * 2, 2),
                "WB": (n_resources_factor * 4, 20),
                "Marking": (n_resources_factor * 4, 1),
                "FT": (n_resources_factor * 2, 1),
            },
            root_cause_resource="WB1",
            # required_material: Dict[str, str] = None,
            merge_after_steps=list(merge_after_steps),
            # n_lots_merge: int = 2,
            split_configs=split_configs,
        ),
        "location": location,
        "window_start": window_start,
        "window_end": window_end,
    }

    if Path(f"output/{scenario_name}/").exists():
        print(f"{scenario_name}: results already exist")
        continue

    count = 0
    for i in range(n_runs):
        event["run_number"] = i
        lambda_response = lambda_client.invoke(
            FunctionName="research-trace-forward-experiments",
            InvocationType="Event",  # RequestResponse
            # LogType="Tail",
            Payload=json.dumps(event).encode(),
        )

        if lambda_response["ResponseMetadata"]["HTTPStatusCode"] == 202:
            count += 1

    print(f"{scenario_name}: {count} invocations")

### Retrieve results from S3

In [ ]:
from botocore.exceptions import ClientError

s3_client = boto3.client("s3")

for i in range(len(doe)):
    merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
    number_of_devices = number_of_devices_values[doe[i][1]]
    n_resources_factor = n_resources_factor_values[doe[i][2]]

    number_of_devices_label = set(str(n) for n in number_of_devices)
    scenario_name = (
        "forward/scenarios/doe/"
        + f"merge_after_steps={'-'.join(merge_after_steps)}"
        + f"+split_after_steps={'-'.join(split_after_steps)}"
        + f"+number_of_devices={'-'.join(sorted(number_of_devices_label))}"
        + f"+n_resources_factor={n_resources_factor}"
    )

    result_files = Path(f"output/{scenario_name}/")
    if not result_files.exists():
        os.mkdir(result_files)

    for i in range(n_runs):
        object_key = os.path.join(
            "research", "trace-forward-experiments", scenario_name, f"run_{i}.json.gz"
        ).replace("\\", "/")
        file_name = os.path.join("output", scenario_name, f"run_{i}.json.gz")

        if Path(file_name).exists():
            print(f"{file_name} already exists")
            continue

        try:
            s3_client.download_file(
                Bucket="574885398813-spill-data", Key=object_key, Filename=file_name
            )
        except ClientError as exception:
            print(f"Failed retrieving: {object_key}: {exception}")

# Process results

## Compute number of devices to recall

In [ ]:
from pandas import DataFrame


def compute_n_recall(probability_dicts, packing_unit_size, threshold) -> DataFrame:
    results = []
    for event_data_file, probability_dict in probability_dicts.items():
        df_trace = DataFrame(probability_dict)

        # Aggregate to get probabilities for target entities
        df_trace_grouped = df_trace.groupby(["entity_source", "entity_target"]).agg(
            {
                "probability": "sum",
                "n_merges": "sum",
                "split_merge_kl": "mean",
                "n_production_steps_aggregations": list,
            }
        )
        df_trace_grouped.reset_index(inplace=True)

        # Remove lists with NaN value
        df_trace_grouped["n_production_steps_aggregations"] = df_trace_grouped[
            "n_production_steps_aggregations"
        ].apply(lambda x: [] if any(not isinstance(i, list) for i in x) else x)

        entities_target = df_trace_grouped["entity_target"].unique()
        n_recall_all = len(entities_target) * packing_unit_size

        # Compute number of devices above threshold
        n_recall_threshold = (
            sum(df_trace_grouped["probability"] > threshold) * packing_unit_size
        )

        results.append(
            {
                "file": event_data_file,
                "probabilities": df_trace_grouped.to_dict(orient="records"),
                "n_recall_all": n_recall_all,
                "n_recall_threshold": n_recall_threshold,
            }
        )

    return DataFrame(results)

In [ ]:
import itertools
import gzip

from pandas import read_csv

threshold = 0.2

processed_scenarios = []  # read_csv("output/forward/combined_report.csv")["scenario_name"].unique()

scenario_names = []
for i in range(len(doe)):
    merge_after_steps, split_after_steps = split_merge_after_steps_values[doe[i][0]]
    number_of_devices = number_of_devices_values[doe[i][1]]
    n_resources_factor = n_resources_factor_values[doe[i][2]]

    number_of_devices_label = set(str(n) for n in number_of_devices)
    scenario_names.append(
        (
            "forward/scenarios/doe/"
            + f"merge_after_steps={'-'.join(merge_after_steps)}"
            + f"+split_after_steps={'-'.join(split_after_steps)}"
            + f"+number_of_devices={'-'.join(sorted(number_of_devices_label))}"
            + f"+n_resources_factor={n_resources_factor}"
        )
    )

scenario_dict = {}
for scenario_name in scenario_names:
    # Skip scenarios that are already processed
    if scenario_name in processed_scenarios:
        continue

    try:
        with open(scenario_name + ".json") as f:
            config = json.load(f)
    except FileNotFoundError:
        print("Skip: ", scenario_name)
        continue

    packing_unit_size = config["packing_unit_size"]

    print(scenario_name)

    probability_dicts = {}
    for result_file in Path(f"output/{scenario_name}/").glob("*.json.gz"):
        with gzip.open(result_file) as f:
            probability_dicts[result_file] = json.load(f)

    scenario_dict[scenario_name] = compute_n_recall(
        probability_dicts, packing_unit_size, threshold
    )

print(len(scenario_dict))

In [ ]:
from pandas import concat

df_combined = DataFrame()
# df_combined = read_csv(f"output/forward/combined_report_doe{len(doe)}.csv")
# df_combined.drop(index=df_combined[df_combined["scenario_name"].str.contains("=Marking")].index, inplace=True)
# df_combined["probabilities"] = df_combined["probabilities"].apply(json.loads)

for k, v in scenario_dict.items():
    v["scenario_name"] = k
    df_combined = concat([df_combined, v])

df_combined["n_recall_diff"] = (
    df_combined["n_recall_all"] - df_combined["n_recall_threshold"]
)

df_combined["probabilities"] = df_combined["probabilities"].apply(json.dumps)
df_combined.to_csv(f"output/forward/combined_report_doe{len(doe)}_threshold={threshold}.csv", index=False)

## Visualize results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import isna, read_csv, set_option
from scipy.stats import pearsonr

set_option("display.max_colwidth", None)

len_doe = 1215
threshold = 0.8
df_combined = read_csv(f"output/forward/combined_report_doe{len_doe}_threshold={threshold}.csv")

In [ ]:
from pandas import notna
from pandas.api.types import CategoricalDtype


def combine_steps(row):
    steps_order = ["WT", "DB", "WB", "Marking", "FT"]
    merge_steps = (
        row["merge_after_steps"].split("-") if notna(row["merge_after_steps"]) else []
    )
    split_steps = (
        row["split_after_steps"].split("-") if notna(row["split_after_steps"]) else []
    )

    combined = []
    for step in steps_order:
        if step in merge_steps:
            combined.append(f"{step}-merge")
        elif step in split_steps:
            combined.append(f"{step}-split")
        else:
            combined.append(f"{step}")

    return "-".join(combined)


def merge_split_order(row, first):
    after = "split" if first == "merge" else "merge"
    i_first = row.find(first)
    i_after = row.find(after)

    return (i_first != -1) & (i_first < i_after)


for parameter in [
    "merge_after_steps",
    "split_after_steps",
    "number_of_devices",
    "n_resources_factor",
]:
    df_combined[parameter] = df_combined["scenario_name"].str.extract(
        rf"{parameter}=([A-Za-z0-9-]+)"
    )

df_combined.replace("25-125", "125-25", inplace=True)

# df_combined["n_resources_factor"] = df_combined["n_resources_factor"].astype(int)

df_combined["merge_split_steps"] = df_combined.apply(combine_steps, axis=1)
df_combined["merge_before_split"] = df_combined["merge_split_steps"].apply(
    merge_split_order, first="merge"
)
df_combined["split_before_merge"] = df_combined["merge_split_steps"].apply(
    merge_split_order, first="split"
)
df_combined["count_merge"] = df_combined["merge_split_steps"].str.count("merge")
df_combined["count_split"] = df_combined["merge_split_steps"].str.count("split")


cat_types = {
    "number_of_devices": CategoricalDtype(
        categories=["50", "25-75", "10-90", "125-25", "10-170"], ordered=True
    ),
}
for c in df_combined.columns:
    if c in cat_types:
        df_combined[c] = df_combined[c].astype(cat_types[c])

In [ ]:
# If "n_recall_all" is NaN, the root cause entity does not occur in the trace
df_combined[~isna(df_combined["n_recall_all"])].describe()

In [ ]:
df_combined.groupby("scenario_name")[
    ["n_recall_diff", "n_recall_all", "n_recall_threshold"]
].describe().T

In [ ]:
index = [
    "count_merge",
    "count_split",
    "merge_before_split",
    # "split_before_merge",
]
columns = ["number_of_devices", "n_resources_factor"]

# Create a pivot table for interaction profile
interaction_profile = (
    df_combined.sort_values(index)
    .pivot_table(
        values="n_recall_diff",
        index=index,
        columns=columns,
        aggfunc="mean",
        observed=False,
    )
    .loc[:, (["50", "25-75", "10-90", "125-25", "10-170"], ["1", "2", "4", "8", "16"])]
)

# Plot the heatmap
plt.figure(figsize=(20, 15))
sns.heatmap(
    interaction_profile,
    annot=True,
    fmt=".2f",
    cmap="Greens",
    cbar_kws={"label": "Mean $\delta_{diff}$"},
)
plt.title("Interaction Profile for $\delta_{diff}$")
plt.xlabel(" | ".join(columns))
plt.ylabel(" | ".join(index))
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=45, ha="right")
plt.tight_layout()


# Change x and y tick labels separator from "-" to " | "
def replace_last_dash(label):
    parts = label.get_text().rsplit("-", 1)
    return " | ".join(parts) if len(parts) == 2 else label.get_text()


ax = plt.gca()
x_labels = [replace_last_dash(label) for label in ax.get_xticklabels()]
y_labels = [label.get_text().replace("-", " | ") for label in ax.get_yticklabels()]
ax.set_xticklabels(x_labels)
ax.set_yticklabels(y_labels)

plt.savefig(
    "output/forward/figures/interaction_profile-n_recall_diff.svg", bbox_inches="tight"
)

In [ ]:
import pandas as pd

import statsmodels.api as sm

# Prepare the data: drop rows with missing n_recall_diff or any factor
factors = [
    "count_merge",
    "count_split",
    # "merge_before_split",
    "split_before_merge",
    "number_of_devices",
    "n_resources_factor",
]
df_reg = df_combined.dropna(subset=factors + ["n_recall_diff"]).copy()

# Encode categorical variables
df_reg["split_before_merge"] = df_reg["split_before_merge"].astype(int)
df_reg["merge_before_split"] = df_reg["merge_before_split"].astype(int)
df_reg["number_of_devices"] = df_reg["number_of_devices"].astype(str)
df_reg["n_resources_factor"] = df_reg["n_resources_factor"].astype(str)

# Create dummy variables for categorical factors
dummy_factors = ["number_of_devices", "n_resources_factor"]
df_reg = pd.get_dummies(df_reg, columns=dummy_factors, drop_first=False, dtype=int)
df_reg.drop(
    ["number_of_devices_50", "n_resources_factor_1"],
    axis=1,
    inplace=True,
    errors="ignore",
)

dummy_columns = [c for c in df_reg.columns if any(f in c for f in dummy_factors)]

# Define X and y
X = df_reg[dummy_columns]
X = pd.concat([df_reg[["count_merge", "count_split", "split_before_merge"]], X], axis=1)
y = df_reg["n_recall_diff"]

# Add constant for intercept
# X = sm.add_constant(X)

# Fit the model
model = sm.OLS(y, X).fit()

# Print the summary
print(model.summary())

In [ ]:
# Get coefficients and confidence intervals
coefs = model.params
conf = model.conf_int()
conf.columns = ["lower", "upper"]

variables = [
    "count_merge",
    "count_split",
    "split_before_merge",
    "number_of_devices_25-75",
    "number_of_devices_10-90",
    "number_of_devices_125-25",
    "number_of_devices_10-170",
    "n_resources_factor_2",
    "n_resources_factor_4",
    "n_resources_factor_8",
    "n_resources_factor_16",
]

# Prepare DataFrame for plotting
coef_df = coefs.to_frame("coef").join(conf)
coef_df = coef_df.reset_index().rename(columns={"index": "variable"})
coef_df.set_index("variable", inplace=True)

# Plot coefficients with confidence intervals
plt.figure(figsize=(8, 5))
plt.errorbar(
    variables,
    coef_df.loc[variables, "coef"],
    yerr=[coef_df["coef"] - coef_df["lower"], coef_df["upper"] - coef_df["coef"]],
    fmt="o",
    capsize=5,
    color="b",
)
plt.axhline(0, color="grey", linestyle="--")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Coefficient")
plt.title("OLS Coefficients with 95% Confidence Intervals")
plt.tight_layout()

plt.savefig("output/forward/figures/mlp-ols-coefficients.svg", bbox_inches="tight")
plt.savefig("output/forward/figures/mlp-ols-coefficients.eps", bbox_inches="tight");

In [ ]:
plt.figure(figsize=(20, 15))
sns.set_theme(font_scale=1)

df_plot = df_combined[
    df_combined["scenario_name"].str.contains("n_resources_factor=1")
].copy()
# df_plot = df_combined.copy()

x_label = "split_after_steps"
df_plot[x_label] = df_plot["scenario_name"].str.extract(rf"{x_label}=([A-Za-z0-9-]+)")
df_plot[x_label] = df_plot[x_label].fillna("-")

group_label = "number_of_devices"
# group_label = "n_resources_factor"
# group_label = "merge_after_steps"
df_plot[group_label] = df_plot["scenario_name"].str.extract(
    rf"{group_label}=([A-Za-z0-9-]+)"
)

ax = sns.violinplot(
    data=df_plot.dropna(subset=["n_recall_diff"]),
    x=x_label,
    y="n_recall_diff",
    hue=group_label,
)

ax.set_ylabel("$R$")
ax.tick_params(axis="x", rotation=90)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

# plt.grid()

plt.savefig(
    f"output/figures/violin_{x_label}_by_{group_label}.svg", bbox_inches="tight"
)

## Visualize trace

In [ ]:
scenario_name = "backward/scenarios/doe/merge_after_steps=WB+split_after_steps=WT+number_of_devices=50+n_resources_factor=1"

event_log_files = basepath.joinpath(f"logs/{scenario_name}")
if not event_log_files.exists():
    os.mkdir(event_log_files)

simulate.main(
    config_file=f"{scenario_name}.json",
    runtime=10000,
    output_event_log_file=event_log_files.joinpath("run.json"),
)

In [ ]:
from aggregated_traces.utils.visualization import generate_graph_visualization

visualize_ekg_graph_rdf = load_rdf_graph(
    event_log_files.joinpath("run.json"),
    store="Oxigraph",
)

# Insert DF/DP relations
visualize_ekg_graph_rdf = insert_DF_DP(visualize_ekg_graph_rdf)

# Insert fractions on the relations
visualize_ekg_graph_rdf = insert_fractions(visualize_ekg_graph_rdf)

# Create NetworkX graph
visualize_ekg_graph_nx = generate_networkx_di_graph(visualize_ekg_graph_rdf)

generate_graph_visualization(visualize_ekg_graph_nx, base_figure_path="output/check")

In [ ]:
visualize_ekg_graph_rdf.query("""
PREFIX : <http://example.org/def/ekg/aggregated_traces/>

SELECT DISTINCT
  ?event
  ?type
  ?product
  ?sum_amount_in
  (replace(?entity_amount_in, "http://example.org/id/ekg/aggregated_traces/", "") as ?_entity_amount_in)
  ?sum_amount_out
  (replace(?entity_amount_out, "http://example.org/id/ekg/aggregated_traces/", "") as ?_entity_amount_out)
  (?sum_amount_in=?sum_amount_out as ?equal)
WHERE {
  { SELECT ?event ?product ?type (sum(xsd:float(strafter(?entity_amount_in, "|"))) as ?sum_amount_in) (group_concat(?entity_amount_in ; separator=", ") as ?entity_amount_in) {
    # Pipe separator (https://www.rfc-editor.org/rfc/rfc3986)
    { SELECT ?event ?product ?type (concat(str(?entity), "|", str(?amount_in)) as ?entity_amount_in) {
      VALUES ?type {
        :DirectlyFollows_AggregatedEntity
        :DirectlyPrecedes_AggregatedEntity
      }
      ?relation a ?type ;
        :target ?event ;
        :amount ?amount_in ;
        :class ?entity .

      ?entity a :AggregatedEntity .

      OPTIONAL {
        ?relation :class ?product .
        ?product a :Product .
      }
    }}
  } GROUP BY ?event ?product ?type }

  { SELECT ?event ?product ?type (sum(xsd:float(strafter(?entity_amount_out, "|"))) as ?sum_amount_out) (group_concat(?entity_amount_out ; separator=", ") as ?entity_amount_out) {
    { SELECT ?event ?product ?type (concat(str(?entity), "|", str(?amount_out)) as ?entity_amount_out) {
      VALUES ?type {
        :DirectlyFollows_AggregatedEntity
        :DirectlyPrecedes_AggregatedEntity
      }
      ?relation a ?type ;
        :source ?event ;
        :amount ?amount_out ;
        :class ?entity .

      ?entity a :AggregatedEntity .

      OPTIONAL {
        ?relation :class ?product .
        ?product a :Product .
      }
    }}
  } GROUP BY ?event ?product ?type }
}
ORDER BY ?type DESC(?equal)""").bindings